### Ref: https://github.com/Lightning-AI/lightning-thunder/blob/main/notebooks/dev_tutorials/extend.ipynb

This is my practice notebook for learning.

In [15]:
import sys
sys.path.insert(0, '..')
from numbers import Number

import thunder
import thunder.torch as ltorch
from thunder.core.devices import DeviceType
from thunder.core.proxies import TensorProxy
from thunder.core.transforms import grad, put_grads, get_grad

import torch
import numpy as np

torch.manual_seed(42);

In [16]:
from thunder.extend import OperatorExecutor, register_executor

In [17]:
# Register a new operator executor
myex = OperatorExecutor("myex", version="0.1")
register_executor(myex)

thunder.extend.OperatorExecutor('myex')

In [18]:
# Operator executor will use the "multimul" function as a new example operator.
#    This function uses NumPy to perform two multiplications of four inputs
def multimul_impl(
    a: Number | torch.Tensor,
    b: Number | torch.Tensor,
    c: Number | torch.Tensor,
    d: Number | torch.Tensor,) -> tuple[torch.Tensor, torch.Tensor]:
    return np.multiply(a, b), np.multiply(c, d)

In [19]:
a = torch.randn((2, 2))
b = torch.randn((2, 2))
multimul_impl(a, b, a, b)

(tensor([[-0.3781, -0.0240],
         [ 0.5177, -0.1470]]),
 tensor([[-0.3781, -0.0240],
         [ 0.5177, -0.1470]]))

In [20]:
# To let thunder use multimul we neeed to define how it propagates metadata. This can be done by directly defining a "meta function",
# of by defining a traceable "like" function that describes what multimul does in terms of existing thunder operations.
#    The "like" function can be used for metadata propagation AND transforming the new operator, as we'll see below.
def multimul_like(
    a: Number | TensorProxy,
    b: Number | TensorProxy,
    c: Number | TensorProxy,
    d: Number | TensorProxy,
):
    return a * b, c * d

In [21]:
# The register_operator method of operator executor's returns a "Symbol" object for multimul that can be called directly
# from compiled thunder code
multimul = myex.register_operator('multimul', like=multimul_like, fn=multimul_impl)

In [22]:
# Example of calling new multimul symbol
def foo(a, b, c, d):
    return multimul(a, b, c, d)

cfoo = thunder.jit(foo, executors=[myex])
cfoo(a, b, a, b)

(tensor([[-0.3781, -0.0240],
         [ 0.5177, -0.1470]]),
 tensor([[-0.3781, -0.0240],
         [ 0.5177, -0.1470]]))

In [23]:
# The symbol is recorded, like other operations, into thunder's trace
thunder.last_traces(cfoo)[-1]

# Constructed by Unwrap the actual return value
import torch
from thunder.executors.torchex import no_autocast

@torch.no_grad()
@no_autocast
def computation(a, b, c, d):
  # a: "cpu f32[2, 2]"
  # b: "cpu f32[2, 2]"
  # c: "cpu f32[2, 2]"
  # d: "cpu f32[2, 2]"

  # /tmp/ipykernel_15715/2538653247.py:3: 	    return multimul(a, b, c, d)
  (t0, t1) = multimul(a, b, c, d)
    # t0 = ltorch.mul(a, b)  # t0: "cpu f32[2, 2]"
      # t0 = prims.mul(a, b)  # t0: "cpu f32[2, 2]"
    # t1 = ltorch.mul(c, d)  # t1: "cpu f32[2, 2]"
      # t1 = prims.mul(c, d)  # t1: "cpu f32[2, 2]"
  return (t0, t1)

In [24]:
# multimul is even differentiable because its "like" function is differentiable
a.requires_grad_(True)
b.requires_grad_(True)

cfoo_grad = grad(cfoo)
cfoo_grad(a, b, a, b)
print(thunder.last_traces(cfoo_grad)[-1])

a.requires_grad_(False)
b.requires_grad_(False)

# Constructed by Unwrap the actual return value
import torch
from thunder.executors.torchex import no_autocast

@torch.no_grad()
@no_autocast
def computation(a, b, c, d):
  # a: "cpu f32[2, 2]"
  # b: "cpu f32[2, 2]"
  # c: "cpu f32[2, 2]"
  # d: "cpu f32[2, 2]"
  t14 = torch.full((2, 2), 1, device=torch.device("cpu"), dtype=torch.float32)  # t14: "cpu f32[2, 2]"
    # t14 = ltorch.full((2, 2), 1, device=torch.device("cpu"), dtype=torch.float32)  # t14: "cpu f32[2, 2]"
      # t14 = prims.full((2, 2), 1, device=devices.Device("cpu"), dtype=dtypes.float32)  # t14: "cpu f32[2, 2]"
  t15 = torch.full((2, 2), 1, device=torch.device("cpu"), dtype=torch.float32)  # t15: "cpu f32[2, 2]"
    # t15 = ltorch.full((2, 2), 1, device=torch.device("cpu"), dtype=torch.float32)  # t15: "cpu f32[2, 2]"
      # t15 = prims.full((2, 2), 1, device=devices.Device("cpu"), dtype=dtypes.float32)  # t15: "cpu f32[2, 2]"
  t16 = torch.mul(d, t15)  # t16: "cpu f32[2, 2]"
    # t16 = ltorch.mul(d, t15)  # t16: "c

tensor([[-1.1229, -0.1863],
        [ 2.2082, -0.6380]])

In [27]:
# We can tell thunder to execute existing operations using multimul by defining a transform
# from them to mmultimul, and a "checker" function that returns True when the
# transform is valid and False otherwise

# We can translate mul to multimul by ignoring the second multiplication
def mul_to_multimul(a: Number | TensorProxy, b: Number | TensorProxy) -> TensorProxy:
    result, _ = multimul(a, b, 0, 0)
    return result

def mul_to_multimul_checker(a: Number | TensorProxy, b: Number | TensorProxy) -> bool:
    def is_cpu(x: Number | TensorProxy) -> bool:
        if isinstance(a, TensorProxy):
            return a.device.devicetype == DeviceType.CPU
        return True

    return all(is_cpu(x) for x in (a, b))

In [28]:
# the register_implementation method describes how to translate mul to multimul
myex.register_implementation(ltorch.mul, checker=mul_to_multimul_checker, execution_transform=mul_to_multimul)

In [29]:
# Verifies the implementation of mul using multimul, and shows the execution transform
def bar(a, b):
    return a * b

cbar = thunder.jit(bar, executors=[myex])
cbar(a, b)
thunder.last_traces(cbar)[-1]

# Constructed by Unwrap the actual return value
import torch
from thunder.executors.torchex import no_autocast

@torch.no_grad()
@no_autocast
def computation(a, b):
  # a: "cpu f32[2, 2]"
  # b: "cpu f32[2, 2]"

  # /tmp/ipykernel_15715/1910031822.py:3: 	    return a * b
  (t2, _) = multimul(a, b, 0, 0)
    # t2 = ltorch.mul(a, b)  # t2: "cpu f32[2, 2]"
      # t2 = prims.mul(a, b)  # t2: "cpu f32[2, 2]"
  return (t2,)

In [30]:
# Execution transforms happen AFTER semantic transforms like grad, so even when computing the grad
#   of mul (which involves two multiplications to compute the grad) we still see multimul in the
#   execution trace
a.requires_grad_(True)
b.requires_grad_(True)

cbar_grad = grad(cbar)
cbar_grad(a, b)
thunder.last_traces(cbar_grad)[-1]

# Constructed by Unwrap the actual return value
import torch
from thunder.executors.torchex import no_autocast

@torch.no_grad()
@no_autocast
def computation(a, b):
  # a: "cpu f32[2, 2]"
  # b: "cpu f32[2, 2]"
  t7 = torch.full((2, 2), 1, device=torch.device("cpu"), dtype=torch.float32)  # t7: "cpu f32[2, 2]"
    # t7 = ltorch.full((2, 2), 1, device=torch.device("cpu"), dtype=torch.float32)  # t7: "cpu f32[2, 2]"
      # t7 = prims.full((2, 2), 1, device=devices.Device("cpu"), dtype=dtypes.float32)  # t7: "cpu f32[2, 2]"
  (t8, _) = multimul(b, t7, 0, 0)
    # t8 = ltorch.mul(b, t7)  # t8: "cpu f32[2, 2]"
      # t8 = prims.mul(b, t7)  # t8: "cpu f32[2, 2]"
  (t9, _) = multimul(a, t7, 0, 0)
    # t9 = ltorch.mul(a, t7)  # t9: "cpu f32[2, 2]"
      # t9 = prims.mul(a, t7)  # t9: "cpu f32[2, 2]"
  del t7
  return [t8, t9]

In [31]:
# In the above grad trace there are two multimuls, and both ignore one of their multiplications.
#   It would be more efficient to perform just one multimul, and we can make this happen
#   by defining a new grad transform for mul that calls multimul once.
#   thunder's grad transforms are defined in a novel way that's not the focus of this notebook,
#   but below we define the grad transform to use multimul.
def mymul_grad(a: TensorProxy, b: TensorProxy) -> TensorProxy:
    fwd = a * b

    g = get_grad(fwd)
    a_grad, b_grad = multimul(b, g, a, g)
    put_grads((a, b), (a_grad, b_grad))

    return fwd

# Re-registers the implementation, including the execution transform and now a grad transform
myex.register_implementation(ltorch.mul, checker=mul_to_multimul_checker, execution_transform=mul_to_multimul, grad_transform=mymul_grad)

In [32]:
# Verifies our new grad transform is used and that a single multimul call is made
cbar_grad = grad(cbar)
cbar_grad(a, b)
thunder.last_traces(cbar_grad)[-1]

# Constructed by Unwrap the actual return value
import torch
from thunder.executors.torchex import no_autocast

@torch.no_grad()
@no_autocast
def computation(a, b):
  # a: "cpu f32[2, 2]"
  # b: "cpu f32[2, 2]"
  t5 = torch.full((2, 2), 1, device=torch.device("cpu"), dtype=torch.float32)  # t5: "cpu f32[2, 2]"
    # t5 = ltorch.full((2, 2), 1, device=torch.device("cpu"), dtype=torch.float32)  # t5: "cpu f32[2, 2]"
      # t5 = prims.full((2, 2), 1, device=devices.Device("cpu"), dtype=dtypes.float32)  # t5: "cpu f32[2, 2]"
  (t2, t3) = multimul(b, t5, a, t5)
    # t2 = ltorch.mul(b, t5)  # t2: "cpu f32[2, 2]"
      # t2 = prims.mul(b, t5)  # t2: "cpu f32[2, 2]"
    # t3 = ltorch.mul(a, t5)  # t3: "cpu f32[2, 2]"
      # t3 = prims.mul(a, t5)  # t3: "cpu f32[2, 2]"
  del t5
  return [t2, t3]

In [33]:
# Some operations may require inputs have particular properties (like be contiguous), or a transform may wish
#   to interleave torch operations with new operations. The transform function supports this. Here
#   we can see an example where the inputs to multimul are made contiguous before it's called
def mul_to_contiguous_multimul(a: Number | TensorProxy, b: Number | TensorProxy) -> TensorProxy:
    a = a.contiguous()
    b = b.contiguous()
    result, _ = multimul(a, b, 0, 0)
    return result

myex.register_implementation(ltorch.mul, checker=mul_to_multimul_checker, execution_transform=mul_to_contiguous_multimul)

In [34]:
# Verifies the new "prologue" for multimul works as expected. Note that the contiguous operations are 
#   executed by PyTorch, and don't have to be executed by your executor
a.requires_grad_(False)
b.requires_grad_(False)

def caz(a, b):
    return a * b
ccaz = thunder.jit(caz, executors=[myex])
ccaz(a, b)
thunder.last_traces(ccaz)[-1]

# Constructed by Unwrap the actual return value
from torch import Tensor
import torch
from thunder.executors.torchex import no_autocast

@torch.no_grad()
@no_autocast
def computation(a, b):
  # a: "cpu f32[2, 2]"
  # b: "cpu f32[2, 2]"

  # /tmp/ipykernel_15715/578292884.py:7: 	    return a * b
  t7 = Tensor.contiguous(a, memory_format=_torch_memory_format_0)  # t7: "cpu f32[2, 2]"
    # t7 = ltorch.contiguous(a, memory_format=_torch_memory_format_0)  # t7: "cpu f32[2, 2]"
      # t7 = prims.stride_order(a, (1, 0))  # t7: "cpu f32[2, 2]"
  t8 = Tensor.contiguous(b, memory_format=_torch_memory_format_0)  # t8: "cpu f32[2, 2]"
    # t8 = ltorch.contiguous(b, memory_format=_torch_memory_format_0)  # t8: "cpu f32[2, 2]"
      # t8 = prims.stride_order(b, (1, 0))  # t8: "cpu f32[2, 2]"
  (t6, _) = multimul(t7, t8, 0, 0)
    # t6 = ltorch.mul(t7, t8)  # t6: "cpu f32[2, 2]"
      # t6 = prims.mul(t7, t8)  # t6: "cpu f32[2, 2]"
  del t7, t8
  return (t6,)

In [36]:
# NVIDIA's APEX cross-entropy executor is a good example of a real-world operator executor. It defines
#   fast forward and backward functions for torch.nn.functional.cross_entropy. We can see its custom
#   fwd and bwd operations below
# NOTE This cell and the following cells require the apex executor be installed to run properly
dtype = torch.float32
device = 'cuda'
logits = torch.randn([2048, 50257], device=device, dtype=ltorch.to_torch_dtype(dtype), requires_grad=False)
labels = torch.randint(0, 50257, [2048], device=device)

from thunder.executors.apexex import apex_ex

def foo(logits, labels):
    return torch.nn.functional.cross_entropy(logits, labels, reduction="mean", ignore_index=-1)
cfoo = thunder.jit(foo, executors=[apex_ex])

RuntimeError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [37]:
# Shows the forward operation
cfoo(logits, labels)
thunder.last_traces(cfoo)[-1]

NameError: name 'logits' is not defined

In [38]:
# Shows APEX's custom forward and backward operations, plus additional PyTorch operations between the two
logits.requires_grad_(True)

cfoo_grad = grad(cfoo)

cfoo_grad(logits, labels)
thunder.last_traces(cfoo_grad)[-1]

NameError: name 'logits' is not defined